# Un seul thermostat peut-il chauffer mille maisons qu'il n'a jamais vues ?

**IFT3700/IFT6758 · Science des données · Labo RL · prototype à tester et recadrer**

Au cours 2, le RL servait à aligner les LLM par préférences. Aujourd'hui, il pilote 1 000 maisons à Montréal en janvier, et on monte une échelle en cinq marches : un seul bâtiment, puis un compromis robuste, puis mémoriser ou reconnaître, puis apprendre en contexte, puis continuer d'apprendre quand la maison change (méta-RL).

> **Prototype.** Les corrections des cinq exercices sont déjà écrites (version solution). La météo est **synthétique** tant que le CSV Open-Meteo n'est pas généré : voir la cellule d'installation. Les agents sont précalculés sur CPU (1 200 itérations de PPO pour les agents à mémoire, une vingtaine de minutes chacun ; une graine) : les chiffres viennent de ce prototype, pas d'un protocole final.

## 0 · Installation

In [ ]:
# Réglages de la séance
SHOW_WIDGETS = True      # curseurs interactifs (sinon, figures statiques)
RUN_TRAINING = False     # True : réentraîne l'agent « classique » en direct (~2 min sur CPU)
METEO_REELLE = False     # True : remplace la météo synthétique par les janviers 2015-2024 d'Open-Meteo (Internet requis)

# Fichiers compagnons : lab_utils.py, agents/, meteo_montreal.csv, tous dans lab_rl_prototype.zip
import os, sys, zipfile
if not os.path.exists("lab_utils.py"):
    archive = "lab_rl_prototype.zip"
    if not os.path.exists(archive):
        try:
            from google.colab import files          # n'existe que dans Colab
        except ImportError:
            raise FileNotFoundError("lab_utils.py introuvable : ouvrez ce notebook depuis le dossier fr/ du lab.")
        print("Fichiers compagnons manquants. Cliquez sur « Choose files » ci-dessous et choisissez lab_rl_prototype.zip.")
        zips = [nom for nom in files.upload() if nom.endswith(".zip")]
        assert zips, "Ce n'est pas un fichier .zip : téléversez lab_rl_prototype.zip."
        archive = zips[0]
    zipfile.ZipFile(archive).extractall(".")
sys.path.insert(0, ".")

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from lab_utils import *
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
torch.set_num_threads(4)

if METEO_REELLE:
    telecharger_open_meteo().to_csv("meteo_montreal.csv", index=False)
meteo = charger_meteo()
print(f"{len(meteo)} heures de météo de janvier, de {meteo.t_ext.min():.0f} à {meteo.t_ext.max():.0f} °C")

## 1 · Le problème : chauffer sans payer la pointe

> **Rappel : le vocabulaire du RL** (nouveau dans ce cours)
> - **État** $s_t$ : ce que l'agent observe à l'heure $t$ (température intérieure et extérieure, prix, heure).
> - **Action** $a_t$ : la puissance de la pompe à chaleur, de 0 à 100 % par paliers de 25 %.
> - **Récompense** $r_t = -\text{coût}_t$, avec $\text{coût}_t = \text{prix}_t \times \text{kWh}_t + \lambda \times \text{inconfort}_t$ (K·h hors de 20-23 °C).
> - **Retour** $G_t = \sum_{k \ge t} r_k$ : tout ce qui reste à gagner jusqu'à la fin de la semaine.
> - **Politique** $\pi(a \mid s)$ : la règle de décision, ce qu'on apprend.

Une maison, c'est deux températures : l'air $T_a$, qui réagit vite, et la masse des murs $T_m$, lente et **invisible** pour le thermostat. Trois paramètres $\xi$ changent d'une maison à l'autre : l'isolation (UA), l'inertie ($C_m$) et la taille de la pompe ($\mu$). Le tarif est en pointe les jours de semaine de 6 h à 9 h et de 16 h à 20 h (0,50 \$/kWh contre 0,07).

In [ ]:
# EXERCICE 1/5 · Un pas d'une heure du modèle à deux températures (Euler, 6 sous-pas de 10 min)
def mon_step(Ta, Tm, u, t_ext, xi, sous_pas=6):
    q_pompe = u * xi["mu"] * xi["UA"] * (T_CONSIGNE - T_DESIGN)     # chaleur fournie (W)
    dt = 3600.0 / sous_pas
    for _ in range(sous_pas):
        # VOTRE CODE ICI : bilan de l'air (pertes, échange avec la masse, pompe, gains) puis de la masse
        dTa = (xi["UA"] * (t_ext - Ta) + HAM * (Tm - Ta) + q_pompe + QINT) / CA
        dTm = HAM * (Ta - Tm) / xi["Cm"]
        Ta, Tm = Ta + dt * dTa, Tm + dt * dTm
    return Ta, Tm, q_pompe / cop(t_ext) / 1000.0

xi_test = tirer_maisons(5, rng=np.random.default_rng(0))
print("Test :", np.allclose(mon_step(20.0, 19.0, 0.5, -10.0, xi_test), step(20.0, 19.0, 0.5, -10.0, xi_test)))

Voici 64 maisons au hasard pendant une semaine, chauffées par un thermostat proportionnel réglé à 21 °C. Les bandes grises sont les heures de pointe.

In [ ]:
xi_test, debuts_test = banc_de_test(meteo, n=256)
d_th = evaluer(thermostat, meteo, xi_test, debuts_test)
pointe = est_pointe(np.arange(H))
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(d_th["Ta"][:64], aspect="auto", cmap="coolwarm", vmin=17, vmax=26)
for t in np.where(np.diff(np.r_[0, pointe]) > 0)[0]:
    ax.axvspan(t - .5, t + 2.5 if t % 24 == 6 else t + 3.5, color="k", alpha=.12, lw=0)
ax.set(xlabel="heure de la semaine", ylabel="maison", title="Le même thermostat dans 64 maisons : certaines gèlent, d'autres surchauffent")
plt.colorbar(im, label="température intérieure (°C)"); plt.show()
print(f"Coût moyen : {cout_par_jour(d_th).mean():.2f} $/jour ; de {cout_par_jour(d_th).min():.2f} à {cout_par_jour(d_th).max():.2f} selon la maison")

## 2 · RL classique : une seule maison

> **Rappel : cours 2.** La régression logistique s'entraîne par $w_j \leftarrow w_j + \eta \sum_i (y_i - \hat y_i)\, x^i_j$.

REINFORCE, c'est la même chose avec deux changements : l'étiquette $y_i$ devient l'action $a_t$ que l'agent a jouée (chauffer ou non), et chaque exemple est pondéré par son retour $G_t$. Une action suivie d'un bon retour devient plus probable.

In [ ]:
# EXERCICE 2/5 · La mise à jour de REINFORCE
def ma_maj(w, X, A, P, G, eta):
    # X : (heures, maisons, 7) caractéristiques ; A : actions 0/1 ; P : probabilités π(chauffer) ; G : retours normalisés
    # VOTRE CODE ICI
    return w + eta * np.einsum("tn,tnj->j", G * (A - P), X) / (X.shape[0] * X.shape[1])

w, hist = reinforce(meteo, iterations=100, eta=5.0, maj=ma_maj, verbose=False)
noms = ["biais", "Ta - 21", "T_ext / 10", "pointe maintenant", "pointe bientôt", "sin heure", "cos heure"]
print(pd.Series(w, index=noms).round(2).to_string())
plt.plot(hist); plt.yscale("log"); plt.xlabel("itération"); plt.ylabel("coût ($/jour)"); plt.title("REINFORCE apprend à chauffer en quelques secondes"); plt.show()

**Question :** lisez les poids comme ceux d'une régression logistique. Pourquoi le poids de « Ta - 21 » est-il fortement négatif ? Et celui de « pointe maintenant » ?

Un agent tout-ou-rien reste grossier. Le même principe, avec cinq niveaux de puissance, une mémoire et l'algorithme PPO, donne l'agent « classique » : entraîné sur cette seule maison, il apprend à préchauffer avant la pointe.

In [ ]:
if RUN_TRAINING:
    agent_classique = Agent(memoire=True)
    entrainer(agent_classique, Maisons(meteo, 256, xi=lambda: maison_nominale(256), seed=1), iterations=80)
else:
    agent_classique, _ = charger_agent("classique")

xi_nom = maison_nominale(256)
for nom, pol in [("thermostat", thermostat), ("thermostat qui préchauffe", thermostat_malin), ("agent classique", agent_classique)]:
    print(f"{nom:28s} {cout_par_jour(evaluer(pol, meteo, xi_nom, debuts_test)).mean():6.2f} $/jour")

def trace(d, i, ax, nom):
    ax.plot(d["Ta"][i], label=nom)

fig, ax = plt.subplots(figsize=(11, 3.5))
for nom, pol in [("thermostat", thermostat), ("agent classique", agent_classique)]:
    ax.plot(evaluer(pol, meteo, xi_nom, debuts_test)["Ta"][0, :72], label=nom)
for t in np.where(est_pointe(np.arange(72)) > 0)[0]:
    ax.axvspan(t - .5, t + .5, color="k", alpha=.1, lw=0)
ax.axhspan(*BANDE, color="g", alpha=.08); ax.set(xlabel="heure", ylabel="°C", title="L'agent monte avant la pointe et redescend pendant"); ax.legend(); plt.show()

## 3 · Robuste : un compromis pour toutes les maisons

L'agent classique ne connaît qu'une maison. Pour en piloter mille, on l'entraîne sur des maisons tirées au hasard : c'est **l'augmentation de données** des cours 1 et 7, appelée en RL *randomisation de domaine*.

In [ ]:
# EXERCICE 3/5 · Tirer n maisons : ξ = nominal × exp(U(-w, w)), w = largeur × log(sqrt(facteur))
def mes_maisons(n, largeur=1.0, rng=np.random.default_rng(0)):
    # VOTRE CODE ICI
    return {k: NOMINAL[k] * np.exp(rng.uniform(-1, 1, n) * largeur * np.log(np.sqrt(FACTEUR[k]))) for k in NOMINAL}

xi_demo = mes_maisons(2000)
print({k: (round(v.min(), 2), round(v.max(), 2)) for k, v in xi_demo.items()})

In [ ]:
oracle, _ = charger_agent("oracle")      # voit ξ et la température de la masse : la borne haute
robuste, _ = charger_agent("robuste")    # sans mémoire, entraîné sur toutes les maisons
res = {}
res["thermostat"] = cout_par_jour(d_th)
res["classique (une maison)"] = cout_par_jour(evaluer(agent_classique, meteo, xi_test, debuts_test))
res["robuste (sans mémoire)"] = cout_par_jour(evaluer(robuste, meteo, xi_test, debuts_test))
res["oracle (voit ξ)"] = cout_par_jour(evaluer(oracle, meteo, xi_test, debuts_test, privilegie=True))
for k, v in res.items():
    print(f"{k:26s} {v.mean():6.2f} $/jour   écart à l'oracle {100 * (v.mean() / res['oracle (voit ξ)'].mean() - 1):+6.1f} %")

z = xi_normalise(xi_test)
ecart = 100 * (res["robuste (sans mémoire)"] / res["oracle (voit ξ)"] - 1)
plt.figure(figsize=(6, 4.5))
sc = plt.scatter(xi_test["UA"], xi_test["mu"], c=ecart, cmap="magma_r", s=25)
plt.colorbar(sc, label="surcoût du robuste face à l'oracle (%)")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("isolation UA (W/K, plus = passoire)"); plt.ylabel("taille de la pompe μ")
plt.title("Le prix de l'ignorance : le même compromis coûte jusqu'à +50 % selon la maison"); plt.show()

## 4 · Le col : pourquoi la mémoire ne sert pas

Donnons une mémoire à l'agent robuste (un GRU : $h_t = f(W[x_t, h_{t-1}] + w_0)$, un MLP qui relit sa propre sortie). Va-t-il apprendre à reconnaître la maison ? Un jouet à deux poids répond. La mémoire écrit $m = c\,\xi + \varepsilon$, l'action la lit $a = b\,m$, l'action idéale est $\xi$ :

$$J(b, c) = \sigma_\xi^2\,(bc - 1)^2 + b^2 \sigma_\varepsilon^2$$

En $(0, 0)$ (la politique robuste), lire ne rapporte rien tant que la mémoire est vide, et écrire ne rapporte rien tant que personne ne lit.

In [ ]:
# EXERCICE 4/5 · Le gradient du jouet
def mon_grad_J(b, c, s_xi=1.0, s_eps=1.0):
    # VOTRE CODE ICI : dJ/db, dJ/dc
    return 2 * s_xi**2 * (b * c - 1) * c + 2 * b * s_eps**2, 2 * s_xi**2 * (b * c - 1) * b

print("Gradient au col (0, 0) :", mon_grad_J(0.0, 0.0), "   Test :", np.allclose(mon_grad_J(.3, -.7), grad_J(.3, -.7)))

def paysage(s_xi=1.0, tete=0.0):
    bs = np.linspace(-1.5, 1.5, 200); B, Cc = np.meshgrid(bs, bs)
    plt.figure(figsize=(5.5, 5)); plt.contourf(B, Cc, np.log(J(B, Cc, s_xi) + tete * (1 - Cc) ** 2 + 1e-3), 30, cmap="Blues_r")
    rng = np.random.default_rng(1)
    for _ in range(20):
        b, c = rng.normal(0, 1e-3, 2)        # on part tout près du col
        traj = [(b, c)]
        for _ in range(150):
            gb, gc = mon_grad_J(b, c, s_xi)
            gc += -2 * tete * (1 - c)                    # la tête de prédiction : un gradient pour écrire, sans passer par la lecture
            b, c = b - 0.03 * gb, c - 0.03 * gc
            traj.append((b, c))
        traj = np.array(traj); plt.plot(traj[:, 0], traj[:, 1], "w-", lw=1); plt.plot(*traj[-1], "wo", ms=4)
    plt.plot(0, 0, "r*", ms=14, label="col : politique robuste"); plt.xlim(-1.5, 1.5); plt.ylim(-1.5, 1.5); plt.xlabel("b : lire la mémoire"); plt.ylabel("c : écrire le bâtiment")
    plt.title(f"σξ = {s_xi:.1f}, tête de prédiction = {tete:.1f}"); plt.legend(loc="lower left"); plt.show()

if SHOW_WIDGETS:
    import ipywidgets as W
    W.interact(paysage, s_xi=W.FloatSlider(1.0, min=0.3, max=2.0, step=0.1, description="largeur σξ"),
               tete=W.FloatSlider(0.0, min=0.0, max=1.0, step=0.1, description="tête"))
else:
    paysage(1.0, 0.0); paysage(1.0, 0.5)

**Question :** élargir la boîte des maisons (σξ) creuse la vallée. Fait-il quitter le col plus vite ? Qu'est-ce qui change quand on allume la tête de prédiction ?

Même expérience sur les vrais agents : un GRU entraîné par PPO **sans** tête de prédiction, et le même **avec** la tête, qui prédit la température de l'heure suivante à partir de la mémoire et de l'action jouée.

In [ ]:
col, _ = charger_agent("col")
contexte, _ = charger_agent("contexte")
lignes = []
for nom, ag in [("GRU sans tête (col)", col), ("GRU avec tête", contexte)]:
    d = evaluer(ag, meteo, xi_test, debuts_test)
    res[nom] = cout_par_jour(d)
    lignes.append(dict(agent=nom, cout=res[nom].mean(),
                       effacer_memoire=effet_effacement(ag, meteo, xi_test, debuts_test),
                       demenager=effet_demenagement(ag, meteo, xi_test, debuts_test),
                       **{f"R2 sonde h{t}": r for t, r in sonde(d, (6, 24, 72, 167)).items()}))
tab = pd.DataFrame(lignes).set_index("agent").round(2)
tab

- **Effacer la mémoire** à l'heure 72 : surcoût sur les 24 heures suivantes (%).
- **Déménager** : on échange les mémoires de deux maisons à l'heure 72 ; surcoût sur les 24 heures suivantes (%).
- **Sonde** : une Ridge de scikit-learn lit $\xi$ dans la mémoire ; $R^2$ sur des maisons jamais vues par la sonde.

**Question :** quel agent *lit* vraiment sa mémoire ? Lequel sait seulement quelque chose sans s'en servir ?

## 5 · Mémoriser ou reconnaître, puis apprendre en contexte

> **Rappel : cours 3 et 4.** Le surapprentissage, c'est mémoriser l'ensemble d'entraînement au lieu de comprendre.

Même agent (GRU avec tête), trois mondes d'entraînement : 4 maisons, 16 maisons, ou une maison nouvelle à chaque épisode. On le teste sur ses maisons d'entraînement et sur des maisons qu'il n'a jamais vues.

In [ ]:
lignes = []
for nom, k in [("memoriser_4", 4), ("memoriser_16", 16), ("contexte", None)]:
    ag, _ = charger_agent(nom)
    if k:
        xi_k = tirer_maisons(k, 1.0, np.random.default_rng(11))
        idx = np.arange(256) % k
        xi_train = {key: v[idx] for key, v in xi_k.items()}
    else:
        xi_train = tirer_maisons(256, 1.0, np.random.default_rng(99))
    ref_train = cout_par_jour(evaluer(oracle, meteo, xi_train, debuts_test, privilegie=True)).mean()
    c_train = cout_par_jour(evaluer(ag, meteo, xi_train, debuts_test)).mean()
    c_test = res.get("GRU avec tête").mean() if not k else cout_par_jour(evaluer(ag, meteo, xi_test, debuts_test)).mean()
    lignes.append(dict(monde=f"{k} maisons" if k else "toujours nouvelles",
                       **{"maisons d'entraînement": 100 * (c_train / ref_train - 1),
                          "maisons nouvelles": 100 * (c_test / res["oracle (voit ξ)"].mean() - 1)}))
gen = pd.DataFrame(lignes).set_index("monde").round(1)
gen.plot.bar(rot=0, figsize=(8, 3.5), ylabel="écart à l'oracle (%)", title="Peu de maisons : il mémorise. Beaucoup : il généralise")
plt.show(); gen

In [ ]:
from sklearn.decomposition import PCA
d = evaluer(contexte, meteo, xi_test, debuts_test)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, t in zip(axs, (2, 96)):
    p = PCA(2).fit_transform(d["m"][:, t])
    s = ax.scatter(p[:, 0], p[:, 1], c=np.log(xi_test["UA"]), cmap="viridis", s=15)
    ax.set(title=f"Mémoire à l'heure {t}", xlabel="CP1", ylabel="CP2")
plt.colorbar(s, ax=axs, label="log isolation UA"); fig.suptitle("Lire dans la mémoire : les maisons se rangent par isolation au fil des heures", y=1.04); plt.show()

## 6 · Méta-RL : quand la maison change en cours de route

Une estimation qui fait la moyenne de **tout** le passé (`expanding().mean()`) réagit de plus en plus lentement : après un saut, elle met autant de temps à se corriger à moitié que l'âge qu'elle avait au moment du saut. Une moyenne qui oublie (`ewm`) garde une demi-vie constante.

In [ ]:
# EXERCICE 5/5 · La demi-vie d'une estimation après un saut
def demi_vie(t0, methode, n=400):
    x = pd.Series(np.r_[np.zeros(t0), np.ones(n - t0)])        # le paramètre saute de 0 à 1 à l'heure t0
    # VOTRE CODE ICI : estimation cumulée ou exponentielle (demi-vie 12 h)
    est = x.expanding().mean() if methode == "cumulée" else x.ewm(halflife=12).mean()
    return int(np.argmax(est.values[t0:] >= 0.5))

print(pd.DataFrame({m: [demi_vie(t0, m) for t0 in (12, 24, 48, 96, 192)] for m in ("cumulée", "exponentielle")},
                   index=pd.Index([12, 24, 48, 96, 192], name="âge au saut (h)")))

Dans le simulateur, la dérive tombe à une heure au hasard : la pompe perd la moitié de sa puissance, ou l'isolation se dégrade d'un facteur 1,8. On compare l'agent entraîné sans dérive à celui entraîné avec.

In [ ]:
derive, _ = charger_agent("derive")
oracle_d, _ = charger_agent("oracle_derive")
env_d = Maisons(meteo, 256, derive=True, seed=123); env_d.reset(xi=xi_test, debuts=debuts_test)
t_saut = env_d.t_saut
apres = lambda d: np.mean([d["cout"][i, t_saut[i]:t_saut[i] + 48].sum() for i in range(len(t_saut))]) / 2
lignes, traces = {}, {}
for nom, ag, priv in [("GRU avec tête, entraîné sans dérive", contexte, False), ("GRU avec tête, entraîné avec dérive", derive, False),
                      ("oracle (voit ξ en direct)", oracle_d, True)]:
    d = evaluer(ag, meteo, xi_test, debuts_test, privilegie=priv, derive=True)
    lignes[nom] = (cout_par_jour(d).mean(), apres(d)); traces[nom] = d["Ta"]
ref = lignes["oracle (voit ξ en direct)"]
derive_tab = pd.DataFrame({k: {"coût sur la semaine ($/jour)": v[0], "coût dans les 48 h après le saut ($/jour)": v[1],
                               "écart à l'oracle après le saut (%)": 100 * (v[1] / ref[1] - 1)} for k, v in lignes.items()}).T.round(2)
derive_tab

In [ ]:
# une maison dont la pompe perd la moitié de sa puissance, où l'écart entre les deux agents est le plus net
noms = list(traces)
inc_apres = lambda Ta, i: np.maximum(BANDE[0] - Ta[i, t_saut[i]:t_saut[i] + 48], 0).sum()
candidats = [j for j in np.where(env_d.type_saut == 0)[0] if inc_apres(traces[noms[2]], j) < 2]
i = max(candidats, key=lambda j: inc_apres(traces[noms[0]], j) - inc_apres(traces[noms[1]], j))
fig, ax = plt.subplots(figsize=(11, 3.5))
for nom, Ta in traces.items():
    ax.plot(Ta[i], label=nom, lw=1.2)
ax.axvline(t_saut[i], color="r", ls="--", label="la pompe perd 50 %"); ax.axhspan(*BANDE, color="g", alpha=.08)
ax.set(xlabel="heure", ylabel="°C", title="Après le saut : qui s'adapte ?"); ax.legend(fontsize=9); plt.show()

## 7 · Synthèse : où chaque agent s'est arrêté

In [ ]:
synthese = pd.DataFrame({"coût ($/jour)": {k: v.mean() for k, v in res.items()}})
synthese["écart à l'oracle (%)"] = 100 * (synthese["coût ($/jour)"] / res["oracle (voit ξ)"].mean() - 1)
synthese.sort_values("coût ($/jour)", ascending=False).round(2)

| Marche | Monde d'entraînement | Agent du jour |
| --- | --- | --- |
| RL classique | une maison | agent classique |
| Robuste | maisons tirées au hasard | MLP sans mémoire ; GRU sans tête, près du col |
| Mémoriser ou reconnaître | 4 ou 16 maisons | GRU avec tête |
| Apprendre en contexte | une maison nouvelle à chaque épisode | GRU avec tête |
| Méta-RL | la maison change pendant l'épisode | GRU avec tête, entraîné avec dérive |

À grande échelle, NOMAD est resté près du col : la mémoire savait un peu, mais la politique ne la lisait pas. C'est la question ouverte de ce labo.